# Watchdog - File System Event Monitoring

This notebook covers the **Watchdog** library, a powerful Python tool for monitoring file system events in real-time. Learn how to detect file creation, deletion, modification, and movement events across directories.

## What is Watchdog?

Watchdog is a Python library that monitors file system events across different platforms (Windows, macOS, Linux). It provides a simple API to detect changes in the file system and trigger custom actions in response.

**Key Features:**
- Cross-platform file system event monitoring
- Detects: created, deleted, modified, and moved events
- Observer pattern for handling events
- Efficient polling and native OS event handling

## 1. Import Required Libraries

In [2]:
pip install watchdog


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import required libraries
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler
import time
import os
from pathlib import Path
from datetime import datetime

print("Watchdog library imported successfully!")
print(f"Watchdog version: {Observer.__module__}")

Watchdog library imported successfully!
Watchdog version: watchdog.observers.fsevents


## 2. Understanding File System Events

Watchdog can detect the following types of file system events:

| Event Type | Description |
|---|---|
| **created** | A new file or directory is created |
| **modified** | An existing file or directory is modified (content or metadata changed) |
| **deleted** | A file or directory is deleted |
| **moved** | A file or directory is moved or renamed |

Each event contains information about:
- **Event type**: The type of change that occurred
- **Source path**: The file or directory path affected
- **Is directory**: Whether the event occurred on a file or directory

## 3. Create a Basic File System Observer

An **Observer** is the core component that monitors the file system for events. Here's how to create and configure one:

In [ ]:
# Create a basic Observer instance
observer = Observer()
print(f"Observer created: {observer}")
print(f"Observer is_alive: {observer.is_alive()}")

# The Observer needs an event handler to do something useful
# We'll associate a handler with a directory path to watch
print("\nObserver properties:")
print(f"  - Daemon: {observer.daemon}")
print(f"  - Is alive: {observer.is_alive()}")

## 4. Implement Custom Event Handlers

Event handlers inherit from `FileSystemEventHandler` and override methods to respond to specific events. Here's how to create a custom handler:

In [ ]:
# Custom event handler class
class MyEventHandler(FileSystemEventHandler):
    """
    Custom handler that responds to file system events
    """
    
    def on_created(self, event):
        """Called when a file or directory is created"""
        if not event.is_directory:
            print(f"📄 File created: {event.src_path}")
        else:
            print(f"📁 Directory created: {event.src_path}")
    
    def on_deleted(self, event):
        """Called when a file or directory is deleted"""
        if not event.is_directory:
            print(f"🗑️  File deleted: {event.src_path}")
        else:
            print(f"🗑️  Directory deleted: {event.src_path}")
    
    def on_modified(self, event):
        """Called when a file or directory is modified"""
        if not event.is_directory:
            print(f"✏️  File modified: {event.src_path}")
        else:
            print(f"✏️  Directory modified: {event.src_path}")
    
    def on_moved(self, event):
        """Called when a file or directory is moved or renamed"""
        if not event.is_directory:
            print(f"➡️  File moved: {event.src_path} → {event.dest_path}")
        else:
            print(f"➡️  Directory moved: {event.src_path} → {event.dest_path}")

# Create an instance
handler = MyEventHandler()
print("Custom event handler created successfully!")

## 5. Monitor a Specific Directory

The Observer needs to be configured to watch a specific directory with an event handler. Here's how to set it up:

In [ ]:
# Create a temporary directory for monitoring
watch_directory = "/tmp/watchdog_test"
os.makedirs(watch_directory, exist_ok=True)

print(f"Watching directory: {watch_directory}")

# Create a new observer
observer = Observer()

# Schedule the handler for the directory
# observer.schedule(event_handler, path, recursive=False)
watch = observer.schedule(handler, watch_directory, recursive=False)

print(f"\nScheduled watch:")
print(f"  - Handler: {handler}")
print(f"  - Path: {watch_directory}")
print(f"  - Recursive: False")
print(f"\nObserver is_alive: {observer.is_alive()}")

## 6. Handle Multiple Event Types - Complete Example

Here's a complete example that demonstrates monitoring multiple event types:

In [ ]:
# Advanced event handler with timestamp and filtering
class AdvancedEventHandler(FileSystemEventHandler):
    def __init__(self, ignore_patterns=None):
        super().__init__()
        self.ignore_patterns = ignore_patterns or []
        self.event_count = 0
    
    def should_ignore(self, path):
        """Check if path should be ignored"""
        for pattern in self.ignore_patterns:
            if pattern in path:
                return True
        return False
    
    def log_event(self, event_type, path, is_directory):
        self.event_count += 1
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        obj_type = "DIR" if is_directory else "FILE"
        print(f"[{timestamp}] [{self.event_count}] {event_type}: {obj_type} - {path}")
    
    def on_created(self, event):
        if not self.should_ignore(event.src_path):
            self.log_event("CREATED", event.src_path, event.is_directory)
    
    def on_deleted(self, event):
        if not self.should_ignore(event.src_path):
            self.log_event("DELETED", event.src_path, event.is_directory)
    
    def on_modified(self, event):
        if not self.should_ignore(event.src_path):
            self.log_event("MODIFIED", event.src_path, event.is_directory)
    
    def on_moved(self, event):
        if not self.should_ignore(event.src_path):
            self.log_event("MOVED", f"{event.src_path} → {event.dest_path}", event.is_directory)
    
    def get_stats(self):
        return f"Total events processed: {self.event_count}"

# Create advanced handler with ignore patterns
advanced_handler = AdvancedEventHandler(ignore_patterns=[".tmp", "__pycache__"])
print("Advanced event handler created!")
print(f"Handlers info: {advanced_handler.get_stats()}")

## 7. Stop and Clean Up Observers

Proper cleanup is important to prevent resource leaks. Here's how to stop the observer:

In [ ]:
# Start the observer
print("Starting observer...")
observer.start()
print(f"Observer started! Is alive: {observer.is_alive()}")

# Let it run for a few seconds
print("\nMonitoring for file system events (10 seconds)...")
time.sleep(10)

# Stop the observer
print("\nStopping observer...")
observer.stop()
print(f"Observer stop() called. Is alive: {observer.is_alive()}")

# Wait for observer to finish
observer.join()
print(f"Observer join() completed. Is alive: {observer.is_alive()}")

print("\nCleanup complete!")

## 8. Practical Examples and Use Cases

**Common Use Cases for Watchdog:**

1. **File Backup Systems**: Automatically backup files when they change
2. **Development Tools**: Reload applications when source files are modified
3. **Data Processing**: Trigger analysis pipelines when new data files arrive
4. **Log Monitoring**: Process logs in real-time as they're written
5. **Directory Synchronization**: Sync directories across systems
6. **Build Systems**: Trigger builds when code changes
7. **Real-time Indexing**: Update search indices when documents change

In [ ]:
# Example 1: CSV Log Monitor - Process CSV files as they're created
class CSVLogMonitor(FileSystemEventHandler):
    """Monitor directory for CSV files and process them"""
    
    def on_created(self, event):
        if event.src_path.endswith('.csv') and not event.is_directory:
            print(f"\n✅ New CSV file detected: {event.src_path}")
            print(f"   Size: {os.path.getsize(event.src_path)} bytes")
            self.process_csv(event.src_path)
    
    def process_csv(self, filepath):
        """Simulate CSV processing"""
        print(f"   Processing: Reading file and analyzing data...")
        # In real scenario, you would use pandas or csv module to read and process
        print(f"   Complete: CSV processed successfully!")

# Example 2: Config File Reloader - Reload config when it changes
class ConfigReloader(FileSystemEventHandler):
    """Reload configuration when config file changes"""
    
    def on_modified(self, event):
        if event.src_path.endswith('.json') and not event.is_directory:
            print(f"\n🔄 Configuration file modified: {event.src_path}")
            self.reload_config(event.src_path)
    
    def reload_config(self, filepath):
        """Simulate configuration reload"""
        print(f"   Reloading configuration...")
        # In real scenario, you would parse JSON and update settings
        print(f"   Configuration reloaded!")

print("Example handlers created:")
print("  1. CSVLogMonitor - Processes CSV files as they're created")
print("  2. ConfigReloader - Reloads configuration on changes")

## 9. Best Practices and Tips

### Best Practices:

1. **Always call `observer.join()`**: Ensures proper cleanup before exiting
   ```python
   observer.stop()
   observer.join()
   ```

2. **Use `try-finally` for cleanup**: Guarantees cleanup even if error occurs
   ```python
   try:
       observer.start()
       observer.join()
   finally:
       observer.stop()
   ```

3. **Ignore unnecessary files**: Filter out temporary files, cache, etc.
   ```python
   if event.src_path.endswith(('.tmp', '.swp')):
       return
   ```

4. **Set `recursive=True` for subdirectories**: Monitor entire directory trees
   ```python
   observer.schedule(handler, path, recursive=True)
   ```

5. **Handle platform differences**: Different OSes report events differently
   - macOS may batch events
   - Windows might report duplicate events
   - Use debouncing if needed

### Performance Considerations:

- **Debouncing**: Combine multiple events from one "logical" change
- **Async Processing**: Don't block event handler with long operations
- **Filtering**: Reduce event noise with pattern matching
- **Thread Safety**: If accessing shared data, use locks

In [ ]:
# Complete production-ready example with debouncing
class DebouncedHandler(FileSystemEventHandler):
    """Handler with debouncing to avoid processing duplicate/rapid events"""
    
    def __init__(self, debounce_time=0.5):
        super().__init__()
        self.debounce_time = debounce_time
        self.last_events = {}
    
    def on_modified(self, event):
        current_time = time.time()
        
        # Check if we've seen this file recently
        if event.src_path in self.last_events:
            last_time = self.last_events[event.src_path]
            if current_time - last_time < self.debounce_time:
                return  # Ignore, too soon
        
        # Update last event time
        self.last_events[event.src_path] = current_time
        
        if not event.is_directory:
            print(f"File modified (debounced): {event.src_path}")

# Example: Using context manager pattern for safety
def run_observer_safely(watch_path, handler_class):
    """Safe observer execution with proper resource cleanup"""
    observer = Observer()
    handler = handler_class()
    
    try:
        observer.schedule(handler, watch_path, recursive=True)
        observer.start()
        print(f"Observer started for: {watch_path}")
        
        # Simulate work
        time.sleep(5)
        
    except KeyboardInterrupt:
        print("KeyboardInterrupt received")
    
    finally:
        observer.stop()
        observer.join()
        print("Observer stopped and cleaned up")

# Demonstrate debounced handler
print("DebouncedHandler created with 0.5 second debounce time")
print("\nKey takeaways:")
print("✓ Always call observer.stop() and observer.join()")
print("✓ Use try-finally for guaranteed cleanup")
print("✓ Implement debouncing for production systems")
print("✓ Filter unnecessary events early")
print("✓ Handle both files and directories appropriately")

## Summary

**Watchdog** is a powerful library for real-time file system monitoring across all major platforms. 

### Key Concepts:
- **Observer**: Main object that monitors file system
- **EventHandler**: Custom class inheriting from `FileSystemEventHandler`
- **Events**: Created, modified, deleted, and moved events
- **Path scheduling**: Associate handlers with directory paths

### Quick Checklist:
- ✅ Import Observer and FileSystemEventHandler
- ✅ Create custom handler class with event methods
- ✅ Instantiate Observer and schedule handler
- ✅ Call `observer.start()`
- ✅ Always call `observer.stop()` and `observer.join()`
- ✅ Use try-finally for proper cleanup

### Resources:
- [Watchdog GitHub](https://github.com/gorakhargosh/watchdog)
- [Official Documentation](https://watchdog.readthedocs.io/)
- Common use cases: Auto-reload, data pipelines, backup systems, file sync

Now you're ready to implement file system monitoring in your Python applications!